## 컴퓨터 비전을 위한 딥러닝

### 합성공 신경망

#### 간단한 컨브넷

In [6]:
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(28,28,1))
x = layers.Conv2D(filters = 32, kernel_size = 3, activation = "relu")(inputs) # 32개의 feature map, 3x3 커널, ReLU 활성화 함수
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters = 64, kernel_size = 3, activation = "relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters = 128, kernel_size = 3, activation = "relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation = "softmax")(x)
model = keras.Model(inputs = inputs, outputs = outputs)

#### summary() 메서드로 모델 정보와 구조 출력

In [7]:
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_9 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_6 (MaxPooling  (None, 13, 13, 32)       0         
 2D)                                                             
                                                                 
 conv2d_10 (Conv2D)          (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_7 (MaxPooling  (None, 5, 5, 64)         0         
 2D)                                                             
                                                                 
 conv2d_11 (Conv2D)          (None, 3, 3, 128)         7385

#### Mnist 이미지에서 컨브넷 훈련하기

In [8]:
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype("float32") / 255
model.compile(optimizer="rmsprop",
              loss = "sparse_categorical_crossentropy",
              metrics = ["accuracy"])
model.fit(train_images, train_labels, epochs = 5, batch_size = 64)

Epoch 1/5
938/938 [==============================] - 9s 10ms/step - loss: 0.1558 - accuracy: 0.9516
Epoch 2/5
938/938 [==============================] - 9s 9ms/step - loss: 0.0450 - accuracy: 0.9864
Epoch 3/5
938/938 [==============================] - 9s 9ms/step - loss: 0.0310 - accuracy: 0.9901
Epoch 4/5
938/938 [==============================] - 9s 9ms/step - loss: 0.0239 - accuracy: 0.9924
Epoch 5/5
938/938 [==============================] - 9s 10ms/step - loss: 0.0183 - accuracy: 0.9945


#### 컨브넷 평가하기

In [9]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"테스트 정확도: {test_acc:.3f}")

313/313 [==============================] - 1s 3ms/step - loss: 0.0239 - accuracy: 0.9931
테스트 정확도: 0.993


#### 최대 풀링층이 빠진 잘못된 구조의 컨브넷

In [10]:
inputs = keras.Input(shape=(28,28,1))
x = layers.Conv2D(filters = 32, kernel_size = 3, activation = "relu")(inputs)
x = layers.Conv2D(filters = 64, kernel_size = 3, activation = "relu")(x)
x = layers.Conv2D(filters = 128, kernel_size = 3, activation = "relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation = "softmax")(x)
model_no_max_pool = keras.Model(inputs = inputs, outputs = outputs)

In [11]:
model_no_max_pool.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_12 (Conv2D)          (None, 26, 26, 32)        320       
                                                                 
 conv2d_13 (Conv2D)          (None, 24, 24, 64)        18496     
                                                                 
 conv2d_14 (Conv2D)          (None, 22, 22, 128)       73856     
                                                                 
 flatten_3 (Flatten)         (None, 61952)             0         
                                                                 
 dense_3 (Dense)             (None, 10)                619530    
                                                                 
Total params: 712,202
Trainable params: 712,202
Non-trainab

## Notes

##### 합성곱 연산

- 학습된 패턴은 평행 이동 불변성을 가진다. 컨브넷이 이미지의 오른쪽 아래 모서리에서 어떤 패턴을 학습했다면 다른 곳에서도 이 패턴을 인식할 수 있다. 완전 연결 네트워크는 새로운 위치에 나타난 것은 새로운 패턴으로 학습해야 한다. 

- 컨브넷은 패턴의 공간적 계층 구조를 학습할 수 있다. 합성곱 연산은 특성 맵이라고 부르는 랭크-3 텐서에 적용된다. 이 텐서는 2개의 공간 축과 깊이 축으로 구성된다. RGB 이미지는 3개의 컬러 채널을 가지므로 깊이 축의 차원이 3이 된다. MNIST 숫자처럼 흑백 이미지는 깊이 축의 차원이 1입니다. 합성곱 연산은 입력 특성 맵에서 작은 패치들을 추출하고 이런 모든 패치에 같은 변환을 적용하여 출력 특성 맵을 만든다.

##### 경계 구조와 패딩

- 5 x 5 크기의 특성 맵을 생각해 볼때, 3 x 3 크기인 윈도우를 사용할때에 출력 특성 맵은 3 x 3 크기가 된다. 그런데 입력과 동일한 높이와 너비를 가진 출력 특성 맵을 얻고 싶다면 *패딩*을 사용해서, *가장자리에 적절한 개수의 행과 열을 추가*함으로써, 모든 입력 타일에 합성곱 윈도우의 중앙을 위치시킬 수 있다. 

##### 합성곱 스트라이드 이해하기

- 출력 크기에 영향을 미치는 다른 요소는 스트라이드이다. 스트라이드는 간단하게 말해 커널이 한번에 몇 칸 움직일까에 대한 설명이다. 즉, *두번의 연속적인 윈도우 사이의 거리*를 *스트라이드*라고 부른다.

##### 풀링과 다운샘플링

- *다운샘플링*을 사용하는 이유는, 처리할 특성 맵의 가중치 개수를 줄이기 위해서이다. 또한, 연속적인 합성곱 층이 점점 커진 윈도우를 통해 바라보도록 만들어 필터의 공간적인 계층 구조를 구성한다.

- 최대 풀링이 다운샘플링을 할 수 있는 유일한 방법은 아니다. 이미 알고 있듯이, 앞선 합성곱 층에서 스트라이드를 사용할 수 있다. 최댓값을 취하는 최대 풀링 대신에 입력 패치의 채널별 평균값을 계산하여 변환하는 평균 풀링을 사용할 수도 있다.